<a href="https://colab.research.google.com/github/dora-lai/fyp-aspect-financial-news/blob/main/LLM_Performance_On_FiQA_2018.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai datasets scikit-learn

import google.generativeai as genai
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import json
import time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Setup API
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-3.5-flash-lite')

# 2. Load FiQA 2018 test set
print("Loading FiQA 2018 dataset")
dataset = load_dataset("pauri32/fiqa-2018")
df_fiqa = pd.DataFrame(dataset['test'])

# Define true labels based on the thresholds used in Tier A
def convert_score(score):
    if score > 0.1: return 'positive'
    elif score < -0.1: return 'negative'
    else: return 'neutral'

df_fiqa['true_label'] = df_fiqa['sentiment_score'].apply(convert_score)
df_fiqa['sentence'] = df_fiqa['sentence'].astype(str)

# 3. Design Prompt
def build_prompt(text, target=""):
    return f"""
You are an expert financial sentiment analyst.
Analyze the sentiment of the following financial text towards the target entity.

Target: {target if target else 'General Market'}
Text: "{text}"

Respond ONLY with a valid JSON object: {{"sentiment": "positive" | "neutral" | "negative"}}
"""

# 4. Run LLM extraction
preds = []
print(f"Processing {len(df_fiqa)} FiQA records")

for index, row in df_fiqa.iterrows():
    target = row.get('target', '')
    try:
        response = model.generate_content(build_prompt(row['sentence'], target))
        # Clean markdown formatting if present
        clean_text = response.text.strip().replace('```json', '').replace('```', '').strip()
        result = json.loads(clean_text, strict=False)
        preds.append(result.get('sentiment', 'neutral'))
        print(f"[{index+1}/{len(df_fiqa)}] OK: {result.get('sentiment')}")
    except Exception as e:
        print(f"[{index+1}/{len(df_fiqa)}] Error: {e}")
        preds.append('neutral')

    # Sleep to avoid rate limits
    time.sleep(4)

df_fiqa['llm_pred'] = preds

# 5. Evaluate and Compare
print("\n" + "="*40)
print("LLM Performance on FiQA 2018")
print("="*40)
print(f"Overall Accuracy: {accuracy_score(df_fiqa['true_label'], df_fiqa['llm_pred']):.4f}")

# Split by Headline vs Post
if 'format' in df_fiqa.columns:
    print("\n Breakdown by Format")
    for fmt in df_fiqa['format'].unique():
        subset = df_fiqa[df_fiqa['format'] == fmt]
        acc = accuracy_score(subset['true_label'], subset['llm_pred'])
        print(f"{fmt.capitalize()} (n={len(subset)}): Accuracy = {acc:.4f}")

# Save the results to CSV
df_fiqa.to_csv("fiqa_llm_predictions.csv", index=False)
print("\nResults saved to fiqa_llm_predictions.csv")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
goo

/tmp/ipykernel_753/2309377299.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Loading FiQA 2018 dataset


train.csv:   0%|          | 0.00/161k [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/25.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/961 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/102 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

Processing 150 FiQA records
[1/150] OK: positive
[2/150] OK: negative
[3/150] OK: neutral
[4/150] OK: negative
[5/150] OK: neutral
[6/150] OK: negative
[7/150] OK: positive
[8/150] OK: positive
[9/150] OK: negative
[10/150] OK: positive
[11/150] OK: neutral
[12/150] OK: negative
[13/150] OK: negative
[14/150] OK: positive
[15/150] OK: negative
[16/150] OK: positive
[17/150] OK: neutral
[18/150] OK: positive
[19/150] OK: neutral
[20/150] OK: positive
[21/150] OK: positive
[22/150] OK: neutral
[23/150] OK: positive
[24/150] OK: neutral
[25/150] OK: positive
[26/150] OK: positive
[27/150] OK: negative
[28/150] OK: neutral
[29/150] OK: negative
[30/150] OK: positive
[31/150] OK: neutral
[32/150] OK: positive
[33/150] OK: positive
[34/150] OK: negative
[35/150] OK: positive
[36/150] OK: positive
[37/150] OK: neutral
[38/150] OK: neutral
[39/150] OK: positive
[40/150] OK: neutral
[41/150] OK: negative
[42/150] OK: neutral
[43/150] OK: neutral
[44/150] OK: neutral
[45/150] OK: positive
[46/15

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
d = pd.read_csv("fiqa_llm_predictions.csv")
print(len(d), d['llm_pred'].value_counts().to_dict())
print(classification_report(d['true_label'], d['llm_pred'], digits=4, zero_division=0))
print(confusion_matrix(d['true_label'], d['llm_pred'], labels=['positive','neutral','negative']))

150 {'positive': 55, 'negative': 50, 'neutral': 44, 'positive\n': 1}
              precision    recall  f1-score   support

    negative     0.8200    0.8200    0.8200        50
     neutral     0.7045    0.6200    0.6596        50
    positive     0.7273    0.8000    0.7619        50
   positive
     0.0000    0.0000    0.0000         0

    accuracy                         0.7467       150
   macro avg     0.5630    0.5600    0.5604       150
weighted avg     0.7506    0.7467    0.7472       150

[[40  7  3]
 [12 31  6]
 [ 3  6 41]]


In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prf
rows = []
for k in (1, 2, 3):
    d = pd.read_csv(f"llm_run_{k}_merged.csv").dropna(subset=['sentiment', 'sentence']).copy()
    m = d['llm_entity'].notna()           # gold rows where the LLM gave the same entity name
    for c in ['entity', 'aspect', 'sentiment', 'llm_entity', 'llm_aspect', 'llm_sentiment']:
        d[c] = d[c].astype(str).str.strip().str.lower()
    f = lambda y, p: prf(list(y), list(p), labels=['negative', 'neutral', 'positive'],
                         average='macro', zero_division=0)[2]
    rows.append({'run': k, 'n_all': len(d), 'n_matched': int(m.sum()),
                 'entity_acc': m.mean(),
                 'pair_all': ((d.aspect == d.llm_aspect) & (d.sentiment == d.llm_sentiment)).mean(),
                 'aspect_acc_all': (d.aspect == d.llm_aspect).mean(),
                 'aspect_acc_matched': (d.aspect[m] == d.llm_aspect[m]).mean(),
                 'sentF1_all': f(d.sentiment, d.llm_sentiment.where(m, 'missing')),
                 'sentF1_matched': f(d.sentiment[m], d.llm_sentiment[m])})
r = pd.DataFrame(rows).set_index('run')
pd.set_option('display.width', 200)
print(r.round(4)); print("\nmean\n", r.mean().round(4)); print("\nsd\n", r.std().round(4))

     n_all  n_matched  entity_acc  pair_all  aspect_acc_all  aspect_acc_matched  sentF1_all  sentF1_matched
run                                                                                                        
1      216        165      0.7639    0.5694          0.6389              0.8364      0.7445          0.8567
2      216        165      0.7639    0.5509          0.6111              0.8000      0.7431          0.8581
3      216        165      0.7639    0.5556          0.6204              0.8121      0.7495          0.8659

mean
 n_all                 216.0000
n_matched             165.0000
entity_acc              0.7639
pair_all                0.5586
aspect_acc_all          0.6235
aspect_acc_matched      0.8162
sentF1_all              0.7457
sentF1_matched          0.8602
dtype: float64

sd
 n_all                 0.0000
n_matched             0.0000
entity_acc            0.0000
pair_all              0.0096
aspect_acc_all        0.0141
aspect_acc_matched    0.0185
sentF1_all 

In [ ]:
import pandas as pd
d = pd.read_csv('fiqa_llm_predictions.csv')
print(d.columns.tolist())

['sentence', 'snippets', 'target', 'sentiment_score', 'aspects', 'format', 'label', 'true_label', 'llm_pred']


In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, classification_report

d = pd.read_csv('fiqa_llm_predictions.csv')
print(d[['sentiment_score', 'label', 'true_label']].head(10))
print(d['label'].unique(), d['true_label'].unique())

   sentiment_score  label true_label
0            0.520      0   positive
1           -0.443      2   negative
2            0.236      0   positive
3           -0.307      2   negative
4            0.282      0   positive
5           -0.221      2   negative
6           -0.299      2   negative
7            0.362      0   positive
8           -0.347      2   negative
9            0.484      0   positive
[0 2 1] ['positive' 'negative' 'neutral']


In [ ]:
from sklearn.metrics import f1_score, classification_report

d['llm_pred']   = d['llm_pred'].astype(str).str.strip().str.lower()
d['true_label'] = d['true_label'].astype(str).str.strip().str.lower()

labels = ['positive', 'neutral', 'negative']
print(d['llm_pred'].unique())    # exactly 3 labels
print(d['true_label'].value_counts())   # 50 / 50 / 50
print(len(d))                    # 150
print(round(f1_score(d['true_label'], d['llm_pred'], labels=labels, average='macro'), 4))  # expect 0.7448
print(classification_report(d['true_label'], d['llm_pred'], labels=labels, digits=4))

['positive' 'negative' 'neutral']
true_label
positive    50
negative    50
neutral     50
Name: count, dtype: int64
150
0.7448
              precision    recall  f1-score   support

    positive     0.7143    0.8000    0.7547        50
     neutral     0.7045    0.6200    0.6596        50
    negative     0.8200    0.8200    0.8200        50

    accuracy                         0.7467       150
   macro avg     0.7463    0.7467    0.7448       150
weighted avg     0.7463    0.7467    0.7448       150

